In [1]:
import duckdb
con = duckdb.connect("../data/processed/complaints.duckdb")

In [2]:
#Are complaints going up or down?
con.execute("""
    WITH monthly AS (
        SELECT month, COUNT(*) AS complaints
        FROM complaints
        WHERE month BETWEEN '2024-01' AND '2026-10'
        GROUP BY month
    )
    SELECT month,
           complaints,
           LAG(complaints) OVER (ORDER BY month) AS previous_month,
           ROUND((complaints - LAG(complaints) OVER (ORDER BY month)) * 100.0
                 / LAG(complaints) OVER (ORDER BY month), 1) AS pct_change
    FROM monthly
    ORDER BY month
""").df()

,month,complaints,previous_month,pct_change
0,2024-01,625,<NA>,NaN
1,2024-02,795,625,27.2
2,2024-03,943,795,18.6
3,2024-04,880,943,-6.7
4,2024-05,932,880,5.9
5,2024-06,930,932,-0.2
6,2024-07,986,930,6.0
7,2024-08,1280,986,29.8
8,2024-09,1279,1280,-0.1
9,2024-10,1735,1279,35.7


In [3]:
#Who drives credit reporting complaints?
con.execute("""
    SELECT company,
           COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_credit_reporting
    FROM complaints
    WHERE is_credit_reporting
    GROUP BY company
    ORDER BY complaints DESC
    LIMIT 5
""").df()

,company,complaints,pct_of_credit_reporting
0,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",30460,34.4
1,"EQUIFAX, INC.",28237,31.9
2,EXPERIAN INFORMATION SOLUTIONS INC.,26189,29.6
3,LEXISNEXIS,436,0.5
4,"CBC COMPANIES, INC.",326,0.4


In [4]:
#What are the top problems in credit reporting
con.execute("""
    SELECT issue, COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM complaints
    WHERE is_credit_reporting
    GROUP BY issue
    ORDER BY complaints DESC
    LIMIT 5
""").df()

,issue,complaints,pct
0,Incorrect information on your report,50946,57.6
1,Problem with a company's investigation into an...,20344,23.0
2,Improper use of your report,16315,18.4
3,Problem with a company's investigation into an...,330,0.4
4,Unable to get your credit report or credit score,295,0.3


In [5]:
#Outside credit reporting, which products have the most complaints
con.execute("""
    SELECT product, COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_non_credit
    FROM complaints
    WHERE NOT is_credit_reporting
    GROUP BY product
    ORDER BY complaints DESC
""").df()

,product,complaints,pct_of_non_credit
0,Debt collection,5722,51.1
1,Credit card,1330,11.9
2,Checking or savings account,1297,11.6
3,"Money transfer, virtual currency, or money ser...",1114,9.9
4,Mortgage,448,4.0
5,Vehicle loan or lease,434,3.9
6,Student loan,306,2.7
7,"Payday loan, title loan, personal loan, or adv...",275,2.5
8,Prepaid card,178,1.6
9,Debt or credit management,95,0.8


In [6]:
#Top 3 issues with each product
con.execute("""
    WITH counts AS (
        SELECT product, issue, COUNT(*) AS complaints
        FROM complaints
        WHERE NOT is_credit_reporting
        GROUP BY product, issue
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product ORDER BY complaints DESC) AS rank_in_product
        FROM counts
    )
    SELECT product, rank_in_product, issue, complaints
    FROM ranked
    WHERE rank_in_product <= 3
    ORDER BY product, rank_in_product
""").df()

,product,rank_in_product,issue,complaints
0,Checking or savings account,1,Managing an account,708
1,Checking or savings account,2,Problem with a lender or other company chargin...,216
2,Checking or savings account,3,Closing an account,148
3,Credit card,1,Problem with a purchase shown on your statement,329
4,Credit card,2,Problem when making payments,170
5,Credit card,3,Fees or interest,153
6,Debt collection,1,Attempts to collect debt not owed,2897
7,Debt collection,2,False statements or representation,914
8,Debt collection,3,Written notification about debt,882
9,Debt or credit management,1,Charged upfront or unexpected fees,25
